# Роботы умного дома

**Теория многоагентных систем · Заречкин Артём · Б24-507**

Три робота обслуживают заявки на уборку, доставку и осмотр помещений. В доме одна линия зарядки. Сначала разберём модель, затем сравним назначение заявок, порядок зарядки и работу при сбоях.

Для запуска в Colab выберите **Среда выполнения → Выполнить всё**. В конце есть ячейка, где можно выбрать сценарий и посмотреть его трассу.

## ЛР 1. Спецификация и архитектура

- **Дом** хранит очередь и выбирает исполнителя. Он не двигает роботов и не управляет питанием.
- **R1, R2, R3** знают свою комнату, заряд и навыки. Робот считает время работы, принимает посильную задачу и просит зарядку, если энергии не хватает.
- **Зарядка** собирает запросы и выдаёт один слот за такт. **Пульт** создаёт задания и имеет право остановить робота.

Среда дискретная. Комнаты соединены проходами. Переход через один проход стоит один такт и одну единицу энергии. Работа в комнате тоже тратит единицу за такт. Робот резервирует энергию при старте, занимает себя на всё время поездки и работы, а комнату обновляет при завершении. Зарядка моделируется общим ресурсом питания, без отдельной поездки к станции.

Для заявки $j$ робот $i$ оценивает длительность как $e_{ij}=d(p_i,p_j)+s_{ij}$, где $d$ — число проходов, а $s_{ij}$ — длительность работы по навыку. Взять заявку можно, если робот свободен, обладает навыком и его запас энергии не меньше $e_{ij}$.

In [ ]:
from collections import Counter, deque
from dataclasses import dataclass, field
from math import ceil, inf
from random import Random
from statistics import mean

ROOMS = {
    "Прихожая": ("Кухня", "Гостиная", "Кабинет"),
    "Кухня": ("Прихожая",),
    "Гостиная": ("Прихожая", "Спальня"),
    "Кабинет": ("Прихожая",),
    "Спальня": ("Гостиная",),
}
SKILLS = {
    "R1": {"уборка": 4, "доставка": 6},
    "R2": {"доставка": 4, "осмотр": 6},
    "R3": {"осмотр": 4, "уборка": 6},
}
START_ROOMS = ("Кухня", "Гостиная", "Кабинет")
CAPACITY = 18
HOME, POWER, USER = "Дом", "Зарядка", "Пульт"


def distances(origin):
    result = {origin: 0}
    queue = deque([origin])
    while queue:
        room = queue.popleft()
        for neighbor in ROOMS[room]:
            if neighbor not in result:
                result[neighbor] = result[room] + 1
                queue.append(neighbor)
    return result


DISTANCE = {room: distances(room) for room in ROOMS}
DIAMETER = max(max(row.values()) for row in DISTANCE.values())


@dataclass(frozen=True)
class Job:
    id: int
    at: int
    kind: str
    room: str
    deadline: int


@dataclass(frozen=True)
class ChargeRequest:
    robot: str
    slack: int
    since: int
    job: int


@dataclass
class Robot:
    id: str
    room: str
    battery: int
    job: Job | None = None
    until: int = 0
    offline: bool = False
    charge_since: int | None = None
    charge_pending: int = 0

    def status(self, t):
        free = not self.offline and self.job is None and t >= self.until
        return free, self.room, self.battery

    def bid(self, job, t):
        return estimate(self.id, self.status(t), job)

    def start(self, job, t):
        eta = self.bid(job, t)
        if eta is None:
            return False
        self.job, self.until = job, t + eta
        self.battery -= eta
        self.charge_since = None
        return True

    def request_charge(self, jobs, t, rate):
        free, room, battery = self.status(t)
        if free:
            compatible = [j for j in jobs if j.kind in SKILLS[self.id]]
            if compatible:
                job = min(compatible, key=lambda j: (j.deadline, j.id))
                eta = DISTANCE[room][job.room] + SKILLS[self.id][job.kind]
                deficit = eta - battery
                if deficit > 0:
                    if self.charge_since is None:
                        self.charge_since = t
                    slack = job.deadline - t - eta - ceil(deficit / rate) - 1
                    return ChargeRequest(self.id, slack, self.charge_since, job.id)
        self.charge_since = None
        return None


def estimate(robot_id, status, job):
    free, room, battery = status
    work = SKILLS[robot_id].get(job.kind)
    if not free or work is None:
        return None
    eta = DISTANCE[room][job.room] + work
    return eta if eta <= battery else None


@dataclass
class ChargeStation:
    policy: str
    backup: bool
    down: bool = False

    @property
    def rate(self):
        return 2 if self.down else 3

    def grant(self, requests, t):
        if not requests or (self.down and not self.backup):
            return None
        if self.policy == "fifo":
            return min(requests, key=lambda r: (r.since, r.robot))

        def priority(request):
            urgency = max(0, 7 - request.slack) if request.slack >= 0 else 0
            score = urgency + 2 * (t - request.since)
            return -score, request.since, request.robot

        return min(requests, key=priority)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.alpha": 0.24,
})

display(pd.DataFrame(SKILLS).fillna("—").rename_axis("Вид работы, такты"))
display(pd.DataFrame(DISTANCE).rename_axis("Расстояния, проходы"))

**Каналы связи**

| Отправитель → получатель | Что передаётся |
|---|---|
| Пульт → Дом | Новая заявка |
| Пульт → робот | Команда остановки |
| Дом ↔ роботы | Телеметрия или предложения, назначение, подтверждение, завершение |
| Роботы ↔ Зарядка | Запрос, разрешение, ожидание либо недоступность |

Роли и коммуникации также показаны на схемах в отчёте ЛР 1.

## ЛР 2. Распределение задач

**Диспетчер** опрашивает роботов, рассчитывает оценку по их состояниям и назначает робота с минимальным временем. **Контрактная сеть** рассылает CFP, получает рассчитанные самими роботами предложения и отправляет результат переговоров. В обоих режимах при равном времени побеждает робот с меньшим ID. Если раунд не нашёл исполнителя, задача возвращается в конец очереди.

От открытия раунда до решения проходит один такт у диспетчера и два в контрактной сети. Для контроля запускаем диспетчера и с двумя тактами. За такт открывается или продвигается один раунд. Стоимость связи условная — 0,05 за одно адресное сообщение назначения и подтверждения. Сообщения зарядки и завершения считаются в трассе отдельно.

In [ ]:
@dataclass
class Round:
    job: Job
    opened: int
    offers: dict = field(default_factory=dict)
    timeout: int | None = None


class Home:
    def __init__(self, jobs, dispatch="central", charging="fifo", *,
                 protected=False, faults=(), batteries=(13, 13, 13),
                 horizon=145, central_delay=1):
        self.jobs = {job.id: job for job in jobs}
        self.arrivals = deque(sorted(jobs, key=lambda j: (j.at, j.id)))
        self.robots = {
            name: Robot(name, room, battery)
            for name, room, battery in zip(SKILLS, START_ROOMS, batteries)
        }
        self.waiting, self.trace, self.done = deque(), [], {}
        self.round = None
        self.dispatch = dispatch
        self.station = ChargeStation(charging, backup=protected)
        self.protected, self.faults = protected, set(faults)
        self.horizon, self.central_delay = horizon, central_delay
        self.requests, self.grants = Counter(), Counter()
        self.max_wait, self.charge_violations, self.violations = 0, 0, 0

    def record(self, t, event, sender="", receiver="", job=None, detail=""):
        self.trace.append({
            "t": t, "event": event, "from": sender, "to": receiver,
            "job": job, "detail": detail,
        })

    def inject(self, t):
        if t == 8 and "dock" in self.faults:
            self.station.down = True
            self.record(t, "DOCK_DOWN", detail="Основная станция недоступна")
            if self.protected:
                self.record(t, "FALLBACK", detail="Питание переключено на резервную станцию")
        if t == 5 and "stop" in self.faults:
            self.stop(t, sender="R3", target="R1")

    def stop(self, t, sender, target):
        self.record(t, "STOP", sender, target)
        if self.protected and sender != USER:
            self.record(t, "DENIED", detail="Команда STOP разрешена только пульту")
            return
        if sender != USER:
            self.violations += 1
        robot = self.robots[target]
        robot.offline = True
        if robot.job is not None:
            self.waiting.append(robot.job)
            self.record(t, "REQUEUE", job=robot.job.id, detail="Исполнитель отключён")
            robot.job = None
        self.record(t, "OFFLINE", detail=target)

    def finish(self, t):
        for robot in self.robots.values():
            if robot.charge_pending:
                robot.battery += robot.charge_pending
                robot.charge_pending = 0
            if robot.job is not None and robot.until == t:
                job = robot.job
                if job.id in self.done:
                    self.violations += 1
                self.done[job.id] = t
                robot.room, robot.job = job.room, None
                self.record(t, "DONE", robot.id, HOME, job.id)

    def assign(self, t):
        if self.round is None and self.waiting:
            job = self.waiting.popleft()
            self.round = Round(job, t)
            event = "STATUS_REQ" if self.dispatch == "central" else "CFP"
            for robot in self.robots.values():
                self.record(t, event, HOME, robot.id, job.id)
            return
        if self.round is None:
            return

        current = self.round
        job = current.job
        age = t - current.opened
        if current.timeout is not None:
            if t == current.timeout:
                self.record(t, "TIMEOUT", job=job.id, detail="Нет подтверждения")
                self.waiting.append(job)
                self.round = None
            return

        if age == 1:
            for robot in self.robots.values():
                if self.dispatch == "central":
                    status = robot.status(t)
                    self.record(t, "STATUS", robot.id, HOME, job.id, str(status))
                    eta = estimate(robot.id, status, job)
                else:
                    fake = "bid" in self.faults and robot.id == "R3"
                    eta = -1 if fake else robot.bid(job, t)
                    self.record(t, "PROPOSE" if eta is not None else "REFUSE",
                                robot.id, HOME, job.id, eta)
                    minimum = SKILLS[robot.id].get(job.kind, inf)
                    if self.protected and eta is not None and not minimum <= eta <= minimum + DIAMETER:
                        self.record(t, "BAD_BID", job=job.id, detail=f"{robot.id}: ETA={eta}")
                        continue
                if eta is not None:
                    current.offers[robot.id] = eta

        delay = self.central_delay if self.dispatch == "central" else 2
        if age < delay:
            return
        if not current.offers:
            self.waiting.append(job)
            self.round = None
            return

        winner = min(current.offers, key=lambda r: (current.offers[r], r))
        self.record(t, "ASSIGN" if self.dispatch == "central" else "ACCEPT",
                    HOME, winner, job.id, current.offers[winner])
        if self.dispatch == "contract":
            for name in current.offers:
                if name != winner:
                    self.record(t, "REJECT", HOME, name, job.id)
        if self.dispatch == "contract" and "bid" in self.faults and winner == "R3":
            self.violations += 1
            current.timeout = t + 2
            return
        if self.robots[winner].start(job, t):
            self.record(t, "ACK", winner, HOME, job.id)
            self.record(t, "START", job=job.id, detail=winner)
        else:
            self.record(t, "NACK", winner, HOME, job.id)
            self.waiting.append(job)
        self.round = None

    def charge(self, t):
        jobs = list(self.waiting)
        reserved = {}
        if self.round is not None:
            jobs.append(self.round.job)
            reserved = self.round.offers
        rate = self.station.rate
        requests = []
        for robot in self.robots.values():
            if robot.id in reserved:
                robot.charge_since = None
                continue
            request = robot.request_charge(jobs, t, rate)
            if request is not None:
                requests.append(request)
                self.requests[robot.id] += 1
                self.record(t, "CHARGE_REQ", robot.id, POWER, request.job,
                            f"запас={request.slack}, ожидание={t - request.since}")
        winner = self.station.grant(requests, t)
        for request in requests:
            if request != winner:
                self.max_wait = max(self.max_wait, t - request.since + 1)
                self.record(t, "WAIT" if winner else "NO_POWER", POWER,
                            request.robot, request.job)
        if winner is None:
            return
        robot = self.robots[winner.robot]
        robot.charge_pending = min(rate, CAPACITY - robot.battery)
        robot.until = t + 1
        robot.charge_since = None
        self.grants[robot.id] += 1
        dock = "резервная" if self.station.down else "основная"
        self.record(t, "GRANT", POWER, robot.id, winner.job, f"+{robot.charge_pending}, {dock}")

    def run(self):
        for t in range(self.horizon + 1):
            self.finish(t)
            if len(self.done) == len(self.jobs) or t == self.horizon:
                break
            self.inject(t)
            while self.arrivals and self.arrivals[0].at == t:
                job = self.arrivals.popleft()
                self.waiting.append(job)
                self.record(t, "JOB", USER, HOME, job.id)
            self.assign(t)
            self.charge(t)
            active_charges = sum(r.charge_pending > 0 for r in self.robots.values())
            self.charge_violations += active_charges > 1
            for robot in self.robots.values():
                self.charge_violations += not (0 <= robot.battery <=
                                               robot.battery + robot.charge_pending <= CAPACITY)
                self.charge_violations += robot.job is not None and robot.charge_pending > 0
        return self

In [ ]:
DISPATCH_EVENTS = {
    "STATUS_REQ", "STATUS", "ASSIGN", "CFP", "PROPOSE", "REFUSE",
    "ACCEPT", "REJECT", "ACK", "NACK",
}


def workload(seed, load):
    rng = Random(seed)
    count, interval, slack = {
        "обычная": (18, 4, 17),
        "пиковая": (30, 2, 15),
        "зарядка": (21, 3, 11),
    }[load]
    jobs = []
    for i in range(count):
        t = i * interval + rng.randrange(interval)
        kind = rng.choice(("уборка", "доставка", "осмотр"))
        room = rng.choice(tuple(ROOMS))
        jobs.append(Job(i + 1, t, kind, room, t + slack + rng.randrange(7)))
    return jobs


def evaluate(home):
    done = len(home.done)
    elapsed = sorted(t - home.jobs[i].at for i, t in home.done.items())
    on_time = sum(t <= home.jobs[i].deadline for i, t in home.done.items())
    messages = sum(e["event"] in DISPATCH_EVENTS for e in home.trace)
    served = [home.grants[r] / n for r, n in home.requests.items()]
    squares = sum(v * v for v in served)
    return {
        "готово, %": 100 * done / len(home.jobs),
        "в срок, %": 100 * on_time / len(home.jobs),
        "среднее время": mean(elapsed) if elapsed else float("nan"),
        "P90 времени": elapsed[ceil(.9 * done) - 1] if elapsed else float("nan"),
        "сообщения": messages,
        "раунды": sum(e["event"] in ("STATUS_REQ", "CFP") for e in home.trace) // len(home.robots),
        "цена связи": messages * 0.05,
        "кванты": sum(home.grants.values()),
        "ожидание, макс.": home.max_wait,
        "индекс Джейна": sum(served) ** 2 / (len(served) * squares) if squares else float("nan"),
        "нарушения": home.violations + home.charge_violations,
        "тайм-ауты": sum(e["event"] == "TIMEOUT" for e in home.trace),
    }


SEEDS = range(20)
COORD_MODES = {
    "Диспетчер": {"dispatch": "central"},
    "Контрактная сеть": {"dispatch": "contract"},
    "Диспетчер (2 такта)": {"dispatch": "central", "central_delay": 2},
}
CHARGE_MODES = {"FIFO": "fifo", "Арбитраж": "priority"}
FAULT_MODES = {
    "Без сбоев": (),
    "Ложная заявка": ("bid",),
    "Чужая команда": ("stop",),
    "Отказ станции": ("dock",),
    "Все сбои": ("bid", "stop", "dock"),
}
CHARGE_START = (2, 2, 2)
# Одновременная потребность в зарядке и срочный осмотр.
CONFLICT = [
    Job(1, 0, "осмотр", "Кабинет", 7),
    Job(2, 0, "доставка", "Кухня", 22),
    Job(3, 0, "уборка", "Спальня", 24),
]


def coordination_runs():
    for load in ("обычная", "пиковая"):
        for seed in SEEDS:
            jobs = workload(seed, load)
            for name, settings in COORD_MODES.items():
                home = Home(jobs, **settings).run()
                yield {"нагрузка": load, "seed": seed, "режим": name, **evaluate(home)}


def charge_runs():
    for seed in SEEDS:
        jobs = workload(seed, "зарядка")
        for name, charging in CHARGE_MODES.items():
            home = Home(jobs, charging=charging, batteries=CHARGE_START).run()
            yield {"seed": seed, "режим": name, **evaluate(home)}


def fault_runs():
    for seed in SEEDS:
        jobs = workload(seed, "зарядка")
        for name, faults in FAULT_MODES.items():
            for protected in (False, True):
                home = Home(jobs, "contract", "priority", protected=protected,
                            faults=faults, batteries=CHARGE_START).run()
                yield {"seed": seed, "сценарий": name, "защита": protected, **evaluate(home)}

In [ ]:
coord = pd.DataFrame(coordination_runs())
coord_cols = ["готово, %", "в срок, %", "среднее время", "P90 времени",
              "раунды", "сообщения", "цена связи"]
coord_avg = coord.groupby(["нагрузка", "режим"], sort=False)[coord_cols].mean()
display(coord_avg.round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), layout="constrained")
for ax, metric, title in zip(axes, ["среднее время", "сообщения"],
                             ["Время от заявки до завершения, такты",
                              "Сообщения назначения за прогон"]):
    coord_avg[metric].unstack().reindex(columns=COORD_MODES).plot.bar(
        ax=ax, rot=0, color=["#345c84", "#ce7440", "#9eb7ca"], legend=False)
    ax.set_title(title)
    ax.set_xlabel("")
fig.legend(handles=axes[0].containers, labels=list(COORD_MODES),
           loc="outside lower center", ncols=3, frameon=False)
plt.show()

Сравнение выполнено на 20 одинаковых наборах заявок для каждого уровня нагрузки. Время и P90 считаются по завершённым заданиям. Доли выполнения и соблюдения срока считаются по всем заявкам. При равных двух тактах временные результаты совпадают на этих входах, а контрактная сеть тратит дополнительные сообщения на отклонение проигравших предложений.

## ЛР 3. Конфликт за зарядку

Роботы запрашивают энергию для ближайшей по сроку совместимой заявки. На один такт можно выдать только один квант. В режиме **FIFO** выигрывает самый ранний запрос, при равенстве — меньший ID. В режиме **арбитража** станция учитывает запас времени $s$ и возраст запроса $a$. Срочность равна $\max(0,7-s)$, если $s\geq 0$, иначе нулю. Выигрывает максимум суммы срочности и $2a$.

Сначала три заявки приходят одновременно. У осмотра кабинета срок 7, а у R3 в начале только две единицы энергии. Затем проверяем обе политики на 20 потоках заявок. Справедливость оцениваем индексом Джейна для долей обслуженных запросов каждого робота и самым долгим непрерывным ожиданием.

In [ ]:
conflict_results = {}
for name, mode in CHARGE_MODES.items():
    run = Home(CONFLICT, charging=mode, batteries=CHARGE_START, horizon=35).run()
    conflict_results[name] = run

display(pd.DataFrame([
    {"режим": name, "осмотр завершён в такт": run.done[1],
     "срок осмотра": CONFLICT[0].deadline, **evaluate(run)}
    for name, run in conflict_results.items()
])[["режим", "осмотр завершён в такт", "срок осмотра", "в срок, %",
    "ожидание, макс.", "индекс Джейна", "нарушения"]].round(3))

fig, ax = plt.subplots(figsize=(8, 2.1), layout="constrained")
robot_colors = {"R1": "#345c84", "R2": "#ce7440", "R3": "#4d866e"}
for row, (name, run) in enumerate(conflict_results.items()):
    for event in run.trace:
        if event["event"] == "GRANT":
            t, robot = event["t"], event["to"]
            ax.barh(row, 0.94, left=t, height=0.55, color=robot_colors[robot])
            ax.text(t + .47, row, robot, ha="center", va="center", color="white")
ax.set(yticks=range(2), yticklabels=list(CHARGE_MODES), xticks=range(6),
       xlabel="Такт начала зарядки", xlim=(0, 5.2))
ax.invert_yaxis()
plt.show()

charge_data = pd.DataFrame(charge_runs())
charge_cols = ["готово, %", "в срок, %", "среднее время", "P90 времени",
               "ожидание, макс.", "индекс Джейна", "кванты", "нарушения"]
charge_avg = charge_data.groupby("режим", sort=False)[charge_cols].mean()
display(charge_avg.round(3))

В заданном конфликте арбитраж успевает зарядить R3 для срочного осмотра. В серии случайных потоков доля выполнения в срок у арбитража немного ниже, а самое долгое ожидание растёт. Польза правила зависит от того, насколько часто встречаются похожие срочные заявки.

## ЛР 4. Диверсионные испытания

Испытываем контрактную сеть и арбитраж зарядки при одинаковых входах до и после защиты.

| Сценарий | До защиты | После защиты |
|---|---|---|
| R3 обещает время −1 и молчит после выигрыша | Срабатывает тайм-аут, заявка повторно попадает в очередь | Невозможная оценка отбрасывается до выбора исполнителя |
| R3 посылает R1 команду остановки | R1 отключается | Получатель разрешает STOP только пульту |
| В такт 8 отказывает зарядная станция | Новые кванты недоступны | Зарядка переключается на резерв с меньшей мощностью |
| Все три сбоя вместе | Проверка накопления отказов | Проверка остаточных потерь |

In [ ]:
fault_data = pd.DataFrame(fault_runs())
fault_cols = ["готово, %", "в срок, %", "среднее время", "тайм-ауты", "нарушения"]
fault_avg = fault_data.groupby(["сценарий", "защита"], sort=False)[fault_cols].mean()
display(fault_avg.rename(index={False: "выкл.", True: "вкл."}, level="защита").round(2))

share = fault_avg["готово, %"].unstack().reindex(FAULT_MODES)
share.columns = ["Без защиты", "С защитой"]
ax = share.plot.barh(figsize=(8, 3.7), color=["#9eb7ca", "#345c84"], width=.72)
ax.set(xlim=(0, 105), ylabel="", xlabel="Выполнено к такту 145, %")
ax.invert_yaxis()
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(.5, 1.16), ncols=2)
plt.tight_layout()
plt.show()

assert (charge_data["нарушения"] == 0).all()
assert all(run.charge_violations == 0 for run in conflict_results.values())
assert (fault_data.loc[fault_data["защита"], "нарушения"] == 0).all()

In [ ]:
# @title Трасса выбранного сценария
scenario = "Все сбои"  # @param ["Без сбоев", "Ложная заявка", "Чужая команда", "Отказ станции", "Все сбои"]
seed = 0  # @param {type:"integer"}

jobs = workload(seed, "зарядка")
important = {"STOP", "DENIED", "OFFLINE", "BAD_BID", "TIMEOUT",
             "DOCK_DOWN", "FALLBACK", "NO_POWER", "GRANT", "START", "DONE"}
traces = {}
for guarded in (False, True):
    run = Home(jobs, "contract", "priority", protected=guarded,
               faults=FAULT_MODES[scenario], batteries=CHARGE_START).run()
    trace = pd.DataFrame(run.trace)
    trace["job"] = trace["job"].astype("Int64")
    mode = "С защитой" if guarded else "Без защиты"
    traces[mode] = trace
    print(mode, f"· завершено {len(run.done)} из {len(jobs)}")
    display(trace.loc[trace.event.isin(important)].head(16))

Полные трассы выбранного сценария доступны в словаре `traces`. Время в таблице ЛР 4 отсутствует, если за прогон не завершилось ни одной заявки. Источник сообщения известен из модели канала, а проверка ставки отсеивает невозможные по навыку и карте значения. Правдоподобная ложь внутри допустимого диапазона остаётся за границами опыта.